# EED-RR experiments

This notebook reproduces every result of the paper from scratch.

**Setup (once):**
1. Zip this repository and upload it as a Kaggle **Dataset** (Kaggle unzips it automatically).
2. In this notebook: *Add Input* → that dataset.
3. Internet is not needed: the SDSC-SP2 trace ships in `data/` (its licence permits redistribution with
   the copyright notice, which is kept in the file header).
4. *Save Version → Save & Run All* to run it in the background.

**Runtime:** the full matrix is roughly 20–40 CPU-minutes; with Kaggle's 4 cores, expect well under an hour.
Results are written incrementally; re-running resumes where it stopped.

In [ ]:
import glob, os, shutil, subprocess, sys

src = None
for p in glob.glob('/kaggle/input/**/run.py', recursive=True):
    if os.path.exists(os.path.join(os.path.dirname(p), 'eedrr_bench', 'engine.py')):
        src = os.path.dirname(p)
        break
assert src, 'Add the eedrr_bench dataset as an input first (see the setup notes above).'
WORK = '/kaggle/working/eedrr_bench'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(src, WORK, ignore=shutil.ignore_patterns('__pycache__'))
os.chdir(WORK)
sys.path.insert(0, WORK)
print('package copied to', WORK)

## 1. Environment and correctness tests
All tests must pass. They include textbook oracles (FCFS/SJF/RR/SRTF), the EED-RR worked example, the published MARR example, and an exact reproduction of the earlier version's results for partition P1.

In [ ]:
import platform, numpy, scipy, pandas, matplotlib
print('Python', platform.python_version(), '| numpy', numpy.__version__, '| scipy', scipy.__version__,
      '| pandas', pandas.__version__, '| matplotlib', matplotlib.__version__, '| CPUs', os.cpu_count())
r = subprocess.run([sys.executable, 'tests/test_core.py'], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'tests failed - do not use these results'

## 2. Real workload trace (Parallel Workloads Archive, SDSC-SP2 cleaned)

In [ ]:
TRACE_URL = 'https://www.cs.huji.ac.il/labs/parallel/workload/l_sdsc_sp2/SDSC-SP2-1998-4.2-cln.swf.gz'
from eedrr_bench.workloads import read_swf

def usable(path):
    try:
        s, rt = read_swf(path)
        print(path, '->', len(s), 'jobs with positive run time')
        return len(s) > 0
    except Exception as e:
        print('not usable:', path, '-', e)
        return False

trace = None
# 1) the copy shipped in the package (data/), or any trace uploaded as a dataset
candidates = [os.path.join(WORK, 'data', 'SDSC-SP2-1998-4.2-cln.swf.gz')]
candidates += glob.glob('/kaggle/input/**/*.swf.gz', recursive=True) + glob.glob('/kaggle/input/**/*.swf', recursive=True)
for c in candidates:
    if os.path.exists(c) and usable(c):
        trace = c
        break
# 2) last resort: download (the archive server may return an HTML page to scripts)
if trace is None:
    try:
        import urllib.request
        os.makedirs('/kaggle/working/traces', exist_ok=True)
        dest = '/kaggle/working/traces/SDSC-SP2-1998-4.2-cln.swf.gz'
        urllib.request.urlretrieve(TRACE_URL, dest)
        if usable(dest):
            trace = dest
    except Exception as e:
        print('Trace download failed:', e)
if trace is None:
    print('Continuing WITHOUT the trace stage.')


## 3. Run all experiments
Stages: validation → scaling-factor selection (validation only) → test → batch → legacy → trace → switch cost → time series.

In [ ]:
OUT = '/kaggle/working/results'
cmd = [sys.executable, 'run.py', '--config', 'full', '--out', OUT, '--workers', str(os.cpu_count())]
if trace:
    cmd += ['--trace', trace]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, 'experiment run failed'

## 4. Statistics, tables and figures

In [ ]:
r = subprocess.run([sys.executable, 'analyze.py', '--results', OUT], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0

In [ ]:
import json
from IPython.display import Image, Markdown, display
print(json.dumps(json.load(open(f'{OUT}/sf_selection.json')), indent=2))
display(Markdown(open(f'{OUT}/analysis/headline.md', encoding='utf-8').read()))
for f in sorted(glob.glob(f'{OUT}/analysis/figures/fig_*.png')):
    print(os.path.basename(f))
    display(Image(f))

## 5. Download
`eedrr_results.zip` (raw CSVs, tables, TIFF figures, figure data) appears under *Output*.

In [ ]:
shutil.make_archive('/kaggle/working/eedrr_results', 'zip', OUT)
print('wrote /kaggle/working/eedrr_results.zip')